In [1]:
import zipfile
import os

src = "/lakehouse/default/Files/buildmate_raw_data.zip"
dst = "/lakehouse/default/Files/"

with zipfile.ZipFile(src) as z:
    z.extractall(dst)

print("expanded")


StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 3, Finished, Available, Finished, False)

expanded


In [2]:
from pyspark.sql import functions as F

customers_path = "Files/raw/customer_master/*.csv"

df_customers = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(customers_path)
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

print("bronze_customers:", df_customers.count())

df_customers.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze_customers")

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 4, Finished, Available, Finished, False)

bronze_customers: 602


In [3]:
from pyspark.sql import functions as F

depots_path = "Files/raw/depots/*.csv"

df_depots= (
    spark.read
    .option("header","true")
    .option("inferschema","false")
    .csv(depots_path)
    .withColumn("ingested_at",F.current_timestamp())
    .withColumn("source_file",F.input_file_name())
)

print("bronze_depots: ",df_depots.count())

df_depots.write.mode("overwrite").format("delta").saveAsTable("bronze_depots")

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 5, Finished, Available, Finished, False)

bronze_depots:  6


In [4]:
rentals_path = "Files/raw/rentals/*.csv"

df_rentals = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(rentals_path)
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

df_rentals.write.mode("overwrite").format("delta").saveAsTable("bronze_rentals")

print("bronze_rentals:", df_rentals.count())

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 6, Finished, Available, Finished, False)

bronze_rentals: 976


In [5]:
billing_path = "Files/raw/billing/*.csv"

df_billing = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(billing_path)
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

df_billing.write.mode("overwrite").format("delta").saveAsTable("bronze_billing")

print("bronze_billing:", df_billing.count())

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 7, Finished, Available, Finished, False)

bronze_billing: 787


In [6]:
for table_name in [
    "bronze_customers",
    "bronze_depots",
    "bronze_rentals",
    "bronze_billing"
]:
    print("\n" + table_name)
    spark.table(table_name).printSchema()

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 8, Finished, Available, Finished, False)


bronze_customers
root
 |-- CUSTOMER_ID: string (nullable = true)
 |-- CUSTOMER_NAME: string (nullable = true)
 |-- REGISTERED_ON: string (nullable = true)
 |-- KYC_VERIFIED_ON: string (nullable = true)
 |-- CUSTOMER_TYPE: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


bronze_depots
root
 |-- DEPOT_CODE: string (nullable = true)
 |-- DEPOT_NAME: string (nullable = true)
 |-- ZONE: string (nullable = true)
 |-- FLEET_SIZE: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


bronze_rentals
root
 |-- rental_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- depot_code: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- checkout_ts: string (nullable = true)
 |-- checkin_ts: string (nullable = true)
 |-- rental_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable =

In [7]:
for table_name in [
    "bronze_customers",
    "bronze_depots",
    "bronze_rentals",
    "bronze_billing"
]:
    cols = spark.table(table_name).columns

    print(
        table_name,
        "ingested_at" in cols,
        "source_file" in cols
    )

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 9, Finished, Available, Finished, False)

bronze_customers True True
bronze_depots True True
bronze_rentals True True
bronze_billing True True


In [8]:
spark.table("bronze_rentals") \
    .select("source_file") \
    .distinct() \
    .orderBy("source_file") \
    .show(100, truncate=False)

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 10, Finished, Available, Finished, False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|source_file                                                                                                                                                                                          |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://04a65503-c86c-4344-8859-385f135ae85e@onelake.dfs.fabric.microsoft.com/05bda063-7da1-44eb-8331-48dd977aca7a/Files/raw/rentals/rentals_2026-06-01.csv?version=1791612532537?flength=2927       |
|abfss://04a65503-c86c-4344-8859-385f135ae85e@onelake.dfs.fabric.microsoft.com/05bda063-7da1-44eb-8331-48dd977aca7a/Files/raw/rentals/rentals_2026-06-02.csv?version=1791612530425?flength=2696       |


In [9]:
df_rental_files = (
    spark.table("bronze_rentals")
    .select("source_file")
    .distinct()
)

df_rental_files.show(100, truncate=False)

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 11, Finished, Available, Finished, False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|source_file                                                                                                                                                                                          |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://04a65503-c86c-4344-8859-385f135ae85e@onelake.dfs.fabric.microsoft.com/05bda063-7da1-44eb-8331-48dd977aca7a/Files/raw/rentals/rentals_2026-06-01.csv?version=1791612532537?flength=2927       |
|abfss://04a65503-c86c-4344-8859-385f135ae85e@onelake.dfs.fabric.microsoft.com/05bda063-7da1-44eb-8331-48dd977aca7a/Files/raw/rentals/rentals_2026-06-12.csv?version=1791612531239?flength=2850       |


In [10]:
from pyspark.sql import functions as F

df_10_june = (
    spark.table("bronze_rentals")
    .withColumn(
        "file_date",
        F.regexp_extract(
            F.col("source_file"),
            r"rentals_(\d{4}-\d{2}-\d{2})",
            1
        )
    )
    .filter(F.col("file_date") == "2026-06-10")
)

result_10_june = (
    df_10_june
    .groupBy("source_file")
    .count()
    .orderBy("source_file")
)

display(result_10_june)

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a3729ce-68f5-44cf-964a-8c7a999b9dbf)

In [12]:
files_10 = [
    r["source_file"]
    for r in result_10_june.select("source_file").collect()
]

print("Number of 10-June files:", len(files_10))

for f in files_10:
    print(f)

StatementMeta(, 4bf8a027-6b89-4cbc-8a16-2f41a7c0a0a2, 14, Finished, Available, Finished, False)

Number of 10-June files: 2
abfss://04a65503-c86c-4344-8859-385f135ae85e@onelake.dfs.fabric.microsoft.com/05bda063-7da1-44eb-8331-48dd977aca7a/Files/raw/rentals/rentals_2026-06-10.csv?version=1791556829171?flength=2465
abfss://04a65503-c86c-4344-8859-385f135ae85e@onelake.dfs.fabric.microsoft.com/05bda063-7da1-44eb-8331-48dd977aca7a/Files/raw/rentals/rentals_2026-06-10_resend.csv?version=1791556829619?flength=2465


In [11]:
file1 = files_10[0]
file2 = files_10[1]

df_file1 = (
    spark.table("bronze_rentals")
    .filter(F.col("source_file") == file1)
    .drop("source_file", "ingested_at")
)

df_file2 = (
    spark.table("bronze_rentals")
    .filter(F.col("source_file") == file2)
    .drop("source_file", "ingested_at")
)

print("File 1 rows:", df_file1.count())
print("File 2 rows:", df_file2.count())

print(
    "File 1 minus File 2:",
    df_file1.exceptAll(df_file2).count()
)

print(
    "File 2 minus File 1:",
    df_file2.exceptAll(df_file1).count()
)

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 13, Finished, Available, Finished, False)

NameError: name 'files_10' is not defined

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_customers = spark.table("bronze_customers")
bronze_depots = spark.table("bronze_depots")
bronze_rentals = spark.table("bronze_rentals")
bronze_billing = spark.table("bronze_billing")

print("Bronze tables loaded successfully")

for name, df in [
    ("bronze_customers", bronze_customers),
    ("bronze_depots", bronze_depots),
    ("bronze_rentals", bronze_rentals),
    ("bronze_billing", bronze_billing)
]:
    print(f"{name}: {df.count()} rows")

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 14, Finished, Available, Finished, False)

Bronze tables loaded successfully
bronze_customers: 602 rows
bronze_depots: 6 rows
bronze_rentals: 976 rows
bronze_billing: 787 rows


In [13]:
def show_damage(df, column, table_name, key=None):
    print(f"\n--- {table_name}: {column} spellings ---")

    (df.groupBy(column)
       .count()
       .orderBy(F.col("count").desc())
       .show(100, truncate=False))

    if key:
        duplicate_groups = (
            df.groupBy(key)
              .count()
              .filter(F.col("count") > 1)
        )

        print(f"Duplicate {key} groups:",
              duplicate_groups.count())

        print("Rows in excess of one per key:",
              df.count() - df.dropDuplicates([key]).count())


show_damage(
    bronze_customers, "CUSTOMER_TYPE",
    "bronze_customers", "CUSTOMER_ID"
)

show_damage(
    bronze_rentals, "rental_type",
    "bronze_rentals", "rental_id"
)

show_damage(
    bronze_billing, "PAYER_TYPE",
    "bronze_billing"
)

print("\n--- Registered date formats ---")
bronze_customers.select("REGISTERED_ON").show(20, truncate=False)

print("\n--- Billing amount formats ---")
bronze_billing.select("AMOUNT_INR").show(20, truncate=False)

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 15, Finished, Available, Finished, False)


--- bronze_customers: CUSTOMER_TYPE spellings ---
+-------------+-----+
|CUSTOMER_TYPE|count|
+-------------+-----+
|company      |62   |
|Contractor   |58   |
|CORP         |56   |
|Corporate    |51   |
|individual   |51   |
|COMPANY      |48   |
|Contr.       |45   |
|contractor   |45   |
|Individual   |42   |
|CONTRACTOR   |41   |
|Company      |37   |
|IND          |35   |
|INDIVIDUAL   |31   |
+-------------+-----+

Duplicate CUSTOMER_ID groups: 2
Rows in excess of one per key: 2

--- bronze_rentals: rental_type spellings ---
+-----------+-----+
|rental_type|count|
+-----------+-----+
|Standard   |241  |
|STANDARD   |235  |
|standard   |228  |
|Priority   |146  |
|PRIORITY   |126  |
+-----------+-----+

Duplicate rental_id groups: 31
Rows in excess of one per key: 31

--- bronze_billing: PAYER_TYPE spellings ---
+----------+-----+
|PAYER_TYPE|count|
+----------+-----+
|CONTRACT  |105  |
|Corporate |97   |
|Contract  |88   |
|CORPORATE |85   |
|Direct    |79   |
|DIRECT    |76   |

In [14]:

from pyspark.sql import functions as F, Window

# Step 1: Deduplicate customers, retaining the latest ingested row
customer_window = Window.partitionBy(
    "CUSTOMER_ID"
).orderBy(
    F.col("ingested_at").desc_nulls_last()
)

customers_deduped = (
    bronze_customers
    .withColumn("_rn", F.row_number().over(customer_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print("Customers after deduplication:", customers_deduped.count())


# Step 2: Standardize CUSTOMER_TYPE to exactly three values
customer_type_clean = F.upper(F.trim(F.col("CUSTOMER_TYPE")))

customers_typed = customers_deduped.withColumn(
    "CUSTOMER_TYPE",
    F.when(
        customer_type_clean.isin("INDIVIDUAL", "IND"),
        "individual"
    ).when(
        customer_type_clean.isin("CONTRACTOR", "CONTR."),
        "contractor"
    ).when(
        customer_type_clean.isin("COMPANY", "CORPORATE", "CORP"),
        "company"
    ).otherwise(F.lit(None).cast("string"))
)


# Step 3: Parse REGISTERED_ON using two date formats
registered_raw = F.trim(F.col("REGISTERED_ON"))

single_format_date = F.to_date(
    registered_raw, "dd-MM-yyyy"
)

customers_dates = (
    customers_typed
    .withColumn(
        "_registered_single",
        single_format_date
    )
    .withColumn(
        "REGISTERED_ON",
        F.coalesce(
            F.to_date(registered_raw, "dd-MM-yyyy"),
            F.to_date(registered_raw, "yyyy/MM/dd")
        )
    )
)


# Step 4: Count dates that a single-format parse would have nulled
single_format_failures = customers_dates.filter(
    registered_raw.isNotNull()
    & (registered_raw != "")
    & F.col("_registered_single").isNull()
).count()

print(
    "Rows nulled by single-format date parsing:",
    single_format_failures
)


# Step 5: Assert zero date parsing failures for nonblank values
date_parse_failures = customers_dates.filter(
    registered_raw.isNotNull()
    & (registered_raw != "")
    & F.col("REGISTERED_ON").isNull()
).count()

print("Registered date parse failures:", date_parse_failures)

assert date_parse_failures == 0, (
    f"Found {date_parse_failures} registered date parse failures"
)


# Step 6: Create the Silver customers DataFrame
silver_customers = customers_dates.drop("_registered_single")


# Step 7: Validate CUSTOMER_TYPE
print("\nCustomer type counts:")
silver_customers.groupBy("CUSTOMER_TYPE").count().show()

invalid_customer_types = silver_customers.filter(
    F.col("CUSTOMER_TYPE").isNull()
    | ~F.col("CUSTOMER_TYPE").isin(
        "individual", "contractor", "company"
    )
).count()

assert invalid_customer_types == 0, (
    f"Found {invalid_customer_types} invalid customer types"
)

assert silver_customers.select(
    "CUSTOMER_TYPE"
).distinct().count() == 3, (
    "CUSTOMER_TYPE must contain exactly three distinct values"
)


# Step 8: Save the Silver table
silver_customers.write.mode("overwrite").saveAsTable(
    "silver_customers"
)

print("\nSilver customers created successfully")
print("Final row count:", silver_customers.count())
print("Distinct customer types:",
      silver_customers.select("CUSTOMER_TYPE").distinct().count())
print("Date parse failures:", date_parse_failures)
print("Single-format date failures:", single_format_failures)


StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 16, Finished, Available, Finished, False)

Customers after deduplication: 600
Rows nulled by single-format date parsing: 273
Registered date parse failures: 0

Customer type counts:
+-------------+-----+
|CUSTOMER_TYPE|count|
+-------------+-----+
|   contractor|  187|
|      company|  254|
|   individual|  159|
+-------------+-----+


Silver customers created successfully
Final row count: 600
Distinct customer types: 3
Date parse failures: 0
Single-format date failures: 273


In [15]:
silver_depots = (
    bronze_depots
    .withColumn("DEPOT_CODE", F.upper(F.trim(F.col("DEPOT_CODE"))))
    .withColumn("DEPOT_NAME", F.trim(F.col("DEPOT_NAME")))
    .withColumn("ZONE", F.initcap(F.trim(F.col("ZONE"))))
    .withColumn(
        "FLEET_SIZE",
        F.regexp_replace(F.trim(F.col("FLEET_SIZE")), ",", "")
         .cast("int")
    )
    .dropDuplicates(["DEPOT_CODE"])
)

silver_depots.write.mode("overwrite").saveAsTable(
    "silver_depots"
)

print("Silver depots created")
print("Rows:", silver_depots.count())
silver_depots.printSchema()

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 17, Finished, Available, Finished, False)

Silver depots created
Rows: 6
root
 |-- DEPOT_CODE: string (nullable = true)
 |-- DEPOT_NAME: string (nullable = true)
 |-- ZONE: string (nullable = true)
 |-- FLEET_SIZE: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [16]:
from pyspark.sql import functions as F, Window

# Step 1: Deduplicate rentals, keeping the latest ingested row
rental_window = Window.partitionBy(
    "rental_id"
).orderBy(
    F.col("ingested_at").desc_nulls_last()
)

rentals_deduped = (
    bronze_rentals
    .withColumn("_rn", F.row_number().over(rental_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print("Rentals after deduplication:", rentals_deduped.count())

# Step 2: Standardize RENTAL_TYPE
rental_type_clean = F.lower(F.trim(F.col("rental_type")))

rentals_typed = rentals_deduped.withColumn(
    "rental_type",
    F.when(
        rental_type_clean.rlike("priority|prio"),
        "priority"
    ).otherwise("standard")
)

# Step 3: Parse checkout and check-in timestamps
rentals_parsed = (
    rentals_typed
    .withColumn(
        "checkout_ts",
        F.to_timestamp(F.col("checkout_ts"))
    )
    .withColumn(
        "checkin_ts",
        F.to_timestamp(F.col("checkin_ts"))
    )
)

# Step 4: Identify impossible rentals
bad = F.col("checkin_ts") < F.col("checkout_ts")

# Quarantine impossible rentals
silver_rentals_quarantine = rentals_parsed.filter(
    F.coalesce(bad, F.lit(False))
)

# Step 5: Count still-out rentals
careless_dropped = rentals_parsed.filter(
    F.col("checkin_ts").isNull()
).count()

print(
    "Still-out rentals that careless filtering would drop:",
    careless_dropped
)

# Step 6: Correct null-safe filter
silver_rentals = rentals_parsed.filter(
    ~F.coalesce(bad, F.lit(False))
)

# Step 7: Verify column types
print("Silver rentals schema:")
silver_rentals.printSchema()

# Step 8: Overwrite existing Delta tables
spark.sql("DROP TABLE IF EXISTS silver_rentals")
spark.sql("DROP TABLE IF EXISTS silver_rentals_quarantine")

silver_rentals_quarantine.write.mode("overwrite").saveAsTable(
    "silver_rentals_quarantine"
)

silver_rentals.write.mode("overwrite").saveAsTable(
    "silver_rentals"
)

# Step 9: Print results
print("silver_rentals rows:", silver_rentals.count())
print(
    "silver_rentals_quarantine rows:",
    silver_rentals_quarantine.count()
)

print("Rental type counts:")
silver_rentals.groupBy("rental_type").count().show()

# Acceptance checks
assert silver_rentals.count() == 942
assert silver_rentals_quarantine.count() == 3
assert silver_rentals.select("rental_type").distinct().count() == 2
assert careless_dropped == 175

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 18, Finished, Available, Finished, False)

Rentals after deduplication: 945
Still-out rentals that careless filtering would drop: 175
Silver rentals schema:
root
 |-- rental_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- depot_code: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- checkout_ts: timestamp (nullable = true)
 |-- checkin_ts: timestamp (nullable = true)
 |-- rental_type: string (nullable = false)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

silver_rentals rows: 942
silver_rentals_quarantine rows: 3
Rental type counts:
+-----------+-----+
|rental_type|count|
+-----------+-----+
|   priority|  261|
|   standard|  681|
+-----------+-----+



In [17]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# STEP 1: LOAD BRONZE BILLING AND SILVER RENTALS
# ============================================================

bronze_billing = spark.table("bronze_billing")
silver_rentals = spark.table("silver_rentals")

print("Bronze billing rows:", bronze_billing.count())
print("Silver rentals rows:", silver_rentals.count())


# ============================================================
# STEP 2: CLEAN AMOUNT_INR
# ============================================================

# Preserve original text for validation
amount_raw = F.trim(F.col("AMOUNT_INR"))

# Remove currency symbols/text, commas and spaces
amount_clean = F.regexp_replace(
    F.col("AMOUNT_INR"),
    r"(?i)R&#x73;|Rs\.?|₹|,|\s",
    ""
)

# Remove remaining non-numeric characters
amount_clean = F.regexp_replace(
    amount_clean,
    r"[^0-9.\-]",
    ""
)

# Convert to decimal(12,2)
billing_parsed = bronze_billing.withColumn(
    "AMOUNT_INR",
    amount_clean.cast("decimal(12,2)")
)


# ============================================================
# STEP 3: STANDARDIZE PAYER_TYPE
# ============================================================

payer_clean = F.lower(F.trim(F.col("PAYER_TYPE")))

billing_parsed = billing_parsed.withColumn(
    "PAYER_TYPE",
    F.when(
        payer_clean.isin("prepaid"),
        "prepaid"
    ).when(
        payer_clean.isin("direct"),
        "direct"
    ).when(
        payer_clean.isin("contract", "contractor"),
        "contract"
    ).when(
        payer_clean.isin("corporate", "corp"),
        "corporate"
    ).otherwise(payer_clean)
)

print("\nPayer type counts:")
billing_parsed.groupBy("PAYER_TYPE").count().show()


# ============================================================
# STEP 4: ASSERT ZERO AMOUNT PARSING FAILURES
# ============================================================

amount_parse_failures = (
    bronze_billing
    .withColumn(
        "_amount_parsed",
        amount_clean.cast("decimal(12,2)")
    )
    .filter(
        F.col("AMOUNT_INR").isNotNull()
        & (F.trim(F.col("AMOUNT_INR")) != "")
        & F.col("_amount_parsed").isNull()
    )
    .count()
)

print("Amount parsing failures:", amount_parse_failures)

assert amount_parse_failures == 0, (
    f"Expected zero amount parsing failures, found {amount_parse_failures}"
)


# ============================================================
# STEP 5: CHECK RENTAL KEY UNIQUENESS
# ============================================================

duplicate_rental_ids = (
    silver_rentals
    .groupBy("rental_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_rental_id_count = duplicate_rental_ids.count()

print("Duplicate rental_id groups:", duplicate_rental_id_count)

if duplicate_rental_id_count > 0:
    duplicate_rental_ids.show(50, truncate=False)

    raise ValueError(
        "silver_rentals contains duplicate rental_id values. "
        "Fix the rental deduplication before joining billing."
    )


# ============================================================
# STEP 6: LEFT JOIN BILLING TO RENTALS
# ============================================================

b = billing_parsed.alias("b")
r = silver_rentals.alias("r")

billing_joined = b.join(
    r,
    F.col("b.RENTAL_ID") == F.col("r.rental_id"),
    "left"
)

print("\nRows before join:", billing_parsed.count())
print("Rows after left join:", billing_joined.count())


# ============================================================
# STEP 7: CREATE UNMATCHED BILLING SIDE TABLE
# ============================================================

silver_billing_unmatched = (
    billing_joined
    .filter(F.col("r.rental_id").isNull())
    .select("b.*")
)

print(
    "Unmatched billing rows:",
    silver_billing_unmatched.count()
)


# ============================================================
# STEP 8: CREATE MATCHED SILVER BILLING TABLE
# ============================================================

silver_billing = (
    billing_joined
    .filter(F.col("r.rental_id").isNotNull())
    .select(
        "b.*",
        F.col("r.customer_id").alias("customer_id"),
        F.col("r.depot_code").alias("depot_code"),
        F.col("r.asset_id").alias("asset_id"),
        F.col("r.checkout_ts").alias("checkout_ts"),
        F.col("r.checkin_ts").alias("checkin_ts"),
        F.col("r.rental_type").alias("rental_type")
    )
)

print("Matched billing rows:", silver_billing.count())


# ============================================================
# STEP 9: VALIDATE COUNTS BEFORE SAVING
# ============================================================

matched_count = silver_billing.count()
unmatched_count = silver_billing_unmatched.count()
source_count = billing_parsed.count()

print("\n========== PRE-SAVE VALIDATION ==========")
print("Source billing:", source_count)
print("Matched billing:", matched_count)
print("Unmatched billing:", unmatched_count)

# Every source billing row must be accounted for exactly once
assert matched_count + unmatched_count == source_count, (
    "Billing row reconciliation failed"
)

assert matched_count == 779, (
    f"Expected 779 matched billing rows, found {matched_count}"
)

assert unmatched_count == 8, (
    f"Expected 8 unmatched billing rows, found {unmatched_count}"
)


# ============================================================
# STEP 10: DROP OLD TABLES TO AVOID DELTA SCHEMA CONFLICTS
# ============================================================

spark.sql("DROP TABLE IF EXISTS silver_billing")
spark.sql("DROP TABLE IF EXISTS silver_billing_unmatched")


# ============================================================
# STEP 11: SAVE SILVER TABLES
# ============================================================

silver_billing.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_billing")

silver_billing_unmatched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_billing_unmatched")


# ============================================================
# STEP 12: VALIDATE SAVED TABLES
# ============================================================

saved_billing = spark.table("silver_billing")
saved_unmatched = spark.table("silver_billing_unmatched")

final_billing_count = saved_billing.count()
final_unmatched_count = saved_unmatched.count()

print("\n========== FINAL BILLING REPORT ==========")
print("silver_billing:", final_billing_count)
print("silver_billing_unmatched:", final_unmatched_count)

print("\nAMOUNT_INR schema:")
saved_billing.select("AMOUNT_INR").printSchema()

print("\nFinal payer type counts:")
saved_billing.groupBy("PAYER_TYPE").count().show()

assert final_billing_count == 779
assert final_unmatched_count == 8




StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 19, Finished, Available, Finished, False)

Bronze billing rows: 787
Silver rentals rows: 942

Payer type counts:
+----------+-----+
|PAYER_TYPE|count|
+----------+-----+
| corporate|  182|
|    direct|  220|
|   prepaid|  192|
|  contract|  193|
+----------+-----+

Amount parsing failures: 0
Duplicate rental_id groups: 0

Rows before join: 787
Rows after left join: 787
Unmatched billing rows: 8
Matched billing rows: 779

========== PRE-SAVE VALIDATION ==========
Source billing: 787
Matched billing: 779
Unmatched billing: 8

========== FINAL BILLING REPORT ==========
silver_billing: 779
silver_billing_unmatched: 8

AMOUNT_INR schema:
root
 |-- AMOUNT_INR: decimal(12,2) (nullable = true)


Final payer type counts:
+----------+-----+
|PAYER_TYPE|count|
+----------+-----+
| corporate|  177|
|    direct|  220|
|   prepaid|  190|
|  contract|  192|
+----------+-----+



In [18]:
expected_tables = [
    "silver_customers",
    "silver_depots",
    "silver_rentals",
    "silver_rentals_quarantine",
    "silver_billing",
    "silver_billing_unmatched"
]

print("=" * 60)
print("TASK 2: SILVER LAYER ACCEPTANCE REPORT")
print("=" * 60)

for table_name in expected_tables:
    df = spark.table(table_name)
    print(f"{table_name}: {df.count()} rows")

customer_count = spark.table("silver_customers").count()
customer_types = spark.table("silver_customers").select(
    "CUSTOMER_TYPE"
).distinct().count()

rental_count = spark.table("silver_rentals").count()
quarantine_count = spark.table("silver_rentals_quarantine").count()
rental_types = spark.table("silver_rentals").select(
    "rental_type"
).distinct().count()

billing_count = spark.table("silver_billing").count()
unmatched_count = spark.table("silver_billing_unmatched").count()

print("\nValidation results:")
print("Customers = 600:", customer_count == 600)
print("Customer types = 3:", customer_types == 3)
print("Rentals = 942:", rental_count == 942)
print("Quarantine = 3:", quarantine_count == 3)
print("Rental types = 2:", rental_types == 2)
print("Single-format date failures = 273:", single_format_failures == 273)
print("Careless filter would drop 175:", careless_dropped == 175)
print("Billing = 779:", billing_count == 779)
print("Unmatched bills = 8:", unmatched_count == 8)
print("Date parse failures = 0:", date_parse_failures == 0)
print("Amount parse failures = 0:", amount_parse_failures == 0)

assert customer_count == 600
assert customer_types == 3
assert rental_count == 942
assert quarantine_count == 3
assert rental_types == 2
assert single_format_failures == 273
assert careless_dropped == 175
assert billing_count == 779
assert unmatched_count == 8
assert date_parse_failures == 0
assert amount_parse_failures == 0

print("\nALL ACCEPTANCE CRITERIA PASSED")

StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 20, Finished, Available, Finished, False)

TASK 2: SILVER LAYER ACCEPTANCE REPORT
silver_customers: 600 rows
silver_depots: 6 rows
silver_rentals: 942 rows
silver_rentals_quarantine: 3 rows
silver_billing: 779 rows
silver_billing_unmatched: 8 rows

Validation results:
Customers = 600: True
Customer types = 3: True
Rentals = 942: True
Quarantine = 3: True
Rental types = 2: True
Single-format date failures = 273: True
Careless filter would drop 175: True
Billing = 779: True
Unmatched bills = 8: True
Date parse failures = 0: True
Amount parse failures = 0: True

ALL ACCEPTANCE CRITERIA PASSED


In [19]:

from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import date

# ============================================================
# STEP 1: LOAD SILVER TABLES AND DEFINE REPORTING WINDOW
# ============================================================

silver_customers = spark.table("silver_customers")
silver_depots = spark.table("silver_depots")
silver_rentals = spark.table("silver_rentals")
silver_billing = spark.table("silver_billing")

# Use the earliest checkout date as the reporting-window start.
# Change this date if your assignment specifies a fixed window.
min_checkout = (
    silver_rentals
    .select(F.min(F.to_date("checkout_ts")).alias("min_date"))
    .first()["min_date"]
)

assert min_checkout is not None, "No valid checkout dates found"

REPORT_START_DATE = min_checkout
REPORT_END_DATE = date.fromordinal(
    REPORT_START_DATE.toordinal() + 29
)

print("Reporting window start:", REPORT_START_DATE)
print("Reporting window end:", REPORT_END_DATE)
print("Reporting window days:", 30)

window_start = F.lit(REPORT_START_DATE).cast("date")
window_end = F.lit(REPORT_END_DATE).cast("date")


# ============================================================
# STEP 2: GOLD_DIM_CUSTOMER
# Derive tenure_years and tenure_band
# ============================================================

today = F.current_date()

gold_dim_customer = (
    silver_customers
    .withColumn(
        "tenure_years",
        F.when(
            F.col("REGISTERED_ON").isNotNull(),
            F.floor(
                F.datediff(today, F.to_date("REGISTERED_ON")) / 365.25
            ).cast("int")
        )
    )
    .withColumn(
        "tenure_band",
        F.when(
            F.col("tenure_years").isNull(), "Unknown"
        ).when(
            F.col("tenure_years") < 1, "Less than 1 year"
        ).when(
            F.col("tenure_years") < 3, "1-2 years"
        ).when(
            F.col("tenure_years") < 5, "3-4 years"
        ).otherwise("5+ years")
    )
)

gold_dim_customer.write.format("delta").mode(
    "overwrite"
).option("overwriteSchema", "true").saveAsTable(
    "gold_dim_customer"
)

print("gold_dim_customer rows:", gold_dim_customer.count())
gold_dim_customer.groupBy("tenure_band").count().show()


# ============================================================
# STEP 3: GOLD_DIM_DEPOT
# Promote the cleaned Silver depot table as-is
# ============================================================

gold_dim_depot = silver_depots

gold_dim_depot.write.format("delta").mode(
    "overwrite"
).option("overwriteSchema", "true").saveAsTable(
    "gold_dim_depot"
)

print("gold_dim_depot rows:", gold_dim_depot.count())


# ============================================================
# STEP 4: GOLD_DIM_DATE
# Generate exactly 30 calendar dates
# ============================================================

gold_dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                window_start,
                window_end,
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("date")
    )
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .withColumn("day_of_week_number", F.dayofweek("date"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
)

gold_dim_date.write.format("delta").mode(
    "overwrite"
).option("overwriteSchema", "true").saveAsTable(
    "gold_dim_date"
)

print("gold_dim_date rows:", gold_dim_date.count())
gold_dim_date.orderBy("date").show(30, truncate=False)

assert gold_dim_date.count() == 30


# ============================================================
# STEP 5: GOLD_FACT_RENTAL
# Keep returned and still-out rentals
# ============================================================

rentals_base = (
    silver_rentals
    .withColumn("_checkout_date", F.to_date("checkout_ts"))
    .withColumn("_checkin_date", F.to_date("checkin_ts"))
    .withColumn(
        "is_returned",
        F.when(F.col("checkin_ts").isNotNull(), 1).otherwise(0)
    )
    .withColumn(
        "rental_duration_days",
        F.when(
            F.col("checkin_ts").isNotNull(),
            F.datediff(
                F.to_date("checkin_ts"),
                F.to_date("checkout_ts")
            )
        ).otherwise(F.lit(None).cast("int"))
    )
)

# Use check-in date for returned rentals.
# For rentals still out, count rented days through the reporting-window end.
rentals_window = (
    rentals_base
    .withColumn(
        "_effective_end",
        F.least(
            F.coalesce(F.col("_checkin_date"), window_end),
            window_end
        )
    )
    .withColumn(
        "_effective_start",
        F.greatest(F.col("_checkout_date"), window_start)
    )
    .withColumn(
        "asset_days_in_window",
        F.when(
            F.col("_checkout_date").isNull()
            | (F.col("_effective_end") < F.col("_effective_start")),
            F.lit(0)
        ).otherwise(
            F.datediff(
                F.col("_effective_end"),
                F.col("_effective_start")
            ) + 1
        ).cast("int")
    )
)

# Drop helper columns; retain every rental
gold_fact_rental = rentals_window.drop(
    "_checkout_date",
    "_checkin_date",
    "_effective_start",
    "_effective_end"
)

gold_fact_rental.write.format("delta").mode(
    "overwrite"
).option("overwriteSchema", "true").saveAsTable(
    "gold_fact_rental"
)

print("gold_fact_rental rows:", gold_fact_rental.count())

print("\nReturned vs still-out rentals:")
gold_fact_rental.groupBy("is_returned").count().show()

print("\nRental duration null checks:")
gold_fact_rental.groupBy(
    "is_returned"
).agg(
    F.count("*").alias("rows"),
    F.sum(
        F.when(F.col("rental_duration_days").isNull(), 1).otherwise(0)
    ).alias("null_duration_rows")
).show()

print("\nMaximum asset_days_in_window:")
gold_fact_rental.select(
    F.max("asset_days_in_window").alias("max_asset_days_in_window")
).show()


# ============================================================
# STEP 6: GOLD_FACT_BILLING
# Promote parsed Silver bills
# ============================================================

gold_fact_billing = silver_billing

gold_fact_billing.write.format("delta").mode(
    "overwrite"
).option("overwriteSchema", "true").saveAsTable(
    "gold_fact_billing"
)

print("gold_fact_billing rows:", gold_fact_billing.count())


# ============================================================
# STEP 7: ACCEPTANCE TESTS
# ============================================================

customer_count = spark.table("gold_dim_customer").count()
depot_count = spark.table("gold_dim_depot").count()
date_count = spark.table("gold_dim_date").count()
rental_count = spark.table("gold_fact_rental").count()
billing_count = spark.table("gold_fact_billing").count()

still_out = spark.table("gold_fact_rental").filter(
    F.col("is_returned") == 0
)

still_out_count = still_out.count()

still_out_bad = still_out.filter(
    F.col("rental_duration_days").isNotNull()
    | (F.col("is_returned") != 0)
).count()

max_asset_days = (
    spark.table("gold_fact_rental")
    .select(F.max("asset_days_in_window").alias("max_days"))
    .first()["max_days"]
)

print("\n========== GOLD ACCEPTANCE REPORT ==========")
print("gold_dim_customer:", customer_count)
print("gold_dim_depot:", depot_count)
print("gold_dim_date:", date_count)
print("gold_fact_rental:", rental_count)
print("gold_fact_billing:", billing_count)
print("Still-out rentals:", still_out_count)
print("Invalid still-out rows:", still_out_bad)
print("Maximum asset days in window:", max_asset_days)

assert customer_count == 600
assert depot_count == 6
assert date_count == 30
assert rental_count == 942
assert billing_count == 779
assert still_out_bad == 0
assert max_asset_days is None or max_asset_days <= 30

print("\nPASS: All Gold acceptance criteria passed.")


# ============================================================
# STEP 8: OPTIMIZE AND ZORDER THE FACT TABLES
# ============================================================

spark.sql("""
    OPTIMIZE gold_fact_rental
    ZORDER BY (rental_id)
""")

spark.sql("""
    OPTIMIZE gold_fact_billing
    ZORDER BY (RENTAL_ID)
""")

print("OPTIMIZE completed for both fact tables.")


# ============================================================
# STEP 9: SHOW DELTA TABLE HISTORY
# ============================================================

print("\nHistory for gold_fact_rental:")
spark.sql(
    "DESCRIBE HISTORY gold_fact_rental"
).show(20, truncate=False)

print("\nHistory for gold_fact_billing:")
spark.sql(
    "DESCRIBE HISTORY gold_fact_billing"
).show(20, truncate=False)


# ============================================================
# STEP 10: REQUIRED TWO-LINE DESIGN NOTE
# ============================================================

print("""
Design note:
We did not partition these small tables because partitioning adds unnecessary overhead at this scale.
Rental duration is NULL rather than zero when a machine is still out because its return date is unknown.
""")


StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 21, Finished, Available, Finished, False)

Reporting window start: 2026-06-01
Reporting window end: 2026-06-30
Reporting window days: 30
gold_dim_customer rows: 600
+----------------+-----+
|     tenure_band|count|
+----------------+-----+
|        5+ years|  174|
|       1-2 years|  231|
|Less than 1 year|   12|
|       3-4 years|  183|
+----------------+-----+

gold_dim_depot rows: 6
gold_dim_date rows: 30
+----------+--------+----+-------+-----+----------+---+-----------+------------------+----------+
|date      |date_key|year|quarter|month|month_name|day|day_of_week|day_of_week_number|is_weekend|
+----------+--------+----+-------+-----+----------+---+-----------+------------------+----------+
|2026-06-01|20260601|2026|2      |6    |June      |1  |Monday     |2                 |false     |
|2026-06-02|20260602|2026|2      |6    |June      |2  |Tuesday    |3                 |false     |
|2026-06-03|20260603|2026|2      |6    |June      |3  |Wednesday  |4                 |false     |
|2026-06-04|20260604|2026|2      |6    |Jun

In [20]:

# ============================================================
# TASK 4: BUSINESS QUESTIONS USING GOLD TABLES
# ============================================================

# Check the Gold table schemas first
for table_name in [
    "gold_fact_rental",
    "gold_fact_billing",
    "gold_dim_depot"
]:
    print(f"\nSchema: {table_name}")
    spark.table(table_name).printSchema()


# ============================================================
# (a) AVERAGE RENTAL DURATION BY ASSET TYPE
# Returned rentals only
# Required: asset_type, avg_duration_days, rentals
# ============================================================

query_a = """
SELECT
    asset_id AS asset_type,
    ROUND(AVG(rental_duration_days), 2) AS avg_duration_days,
    COUNT(*) AS rentals
FROM gold_fact_rental
WHERE is_returned = 1
GROUP BY asset_id
ORDER BY avg_duration_days DESC
"""

print("\n========== (a) DURATION BY ASSET TYPE ==========")
result_a = spark.sql(query_a)
result_a.show(100, truncate=False)


# ============================================================
# (b) FLEET UTILISATION BY DEPOT
# Required: depot_name, fleet_size, asset_days_used, utilisation_pct
# ============================================================

query_b = """
SELECT
    d.DEPOT_NAME AS depot_name,
    d.FLEET_SIZE AS fleet_size,
    ROUND(COALESCE(SUM(r.asset_days_in_window), 0), 1)
        AS asset_days_used,
    ROUND(
        100.0 * COALESCE(SUM(r.asset_days_in_window), 0)
        / NULLIF(CAST(d.FLEET_SIZE AS DOUBLE) * 30, 0),
        1
    ) AS utilisation_pct
FROM gold_dim_depot d
LEFT JOIN gold_fact_rental r
    ON d.DEPOT_CODE = r.depot_code
GROUP BY
    d.DEPOT_NAME,
    d.FLEET_SIZE
ORDER BY utilisation_pct DESC
"""

print("\n========== (b) FLEET UTILISATION ==========")
result_b = spark.sql(query_b)
result_b.show(100, truncate=False)


# ============================================================
# (c) REVENUE BY PAYER TYPE
# Required: payer_type, bills, revenue, avg_bill
# ============================================================

query_c = """
SELECT
    PAYER_TYPE AS payer_type,
    COUNT(*) AS bills,
    ROUND(SUM(AMOUNT_INR), 0) AS revenue,
    ROUND(AVG(AMOUNT_INR), 0) AS avg_bill
FROM gold_fact_billing
GROUP BY PAYER_TYPE
ORDER BY revenue DESC
"""

print("\n========== (c) REVENUE BY PAYER TYPE ==========")
result_c = spark.sql(query_c)
result_c.show(100, truncate=False)


# ============================================================
# (d) REVENUE BY DEPOT
# Join billing -> rentals -> depot dimension
# Required: depot_name, bills, revenue, revenue_per_rental
# ============================================================

query_d = """
WITH depot_revenue AS (
    SELECT
        d.DEPOT_NAME AS depot_name,
        COUNT(DISTINCT b.BILL_ID) AS bills,
        SUM(b.AMOUNT_INR) AS revenue,
        COUNT(DISTINCT r.rental_id) AS rental_count
    FROM gold_fact_billing b
    INNER JOIN gold_fact_rental r
        ON b.RENTAL_ID = r.rental_id
    INNER JOIN gold_dim_depot d
        ON r.depot_code = d.DEPOT_CODE
    GROUP BY d.DEPOT_NAME
)
SELECT
    depot_name,
    bills,
    ROUND(revenue, 0) AS revenue,
    ROUND(
        revenue / NULLIF(rental_count, 0),
        0
    ) AS revenue_per_rental
FROM depot_revenue
ORDER BY revenue DESC
"""

print("\n========== (d) REVENUE BY DEPOT ==========")
result_d = spark.sql(query_d)
result_d.show(100, truncate=False)


# ============================================================
# (e) SHARE OF RENTALS BY RENTAL TYPE
# Required: rental_type, rentals, pct
# ============================================================

query_e = """
WITH rental_mix AS (
    SELECT
        rental_type,
        COUNT(*) AS rentals
    FROM gold_fact_rental
    GROUP BY rental_type
)
SELECT
    rental_type,
    rentals,
    ROUND(
        100.0 * rentals / SUM(rentals) OVER (),
        1
    ) AS pct
FROM rental_mix
ORDER BY pct DESC
"""

print("\n========== (e) PRIORITY MIX ==========")
result_e = spark.sql(query_e)
result_e.show(100, truncate=False)


# ============================================================
# (f) MACHINES CURRENTLY OUT BY DEPOT
# Required: depot_name, currently_out
# ============================================================

query_f = """
SELECT
    d.DEPOT_NAME AS depot_name,
    COUNT(r.rental_id) AS currently_out
FROM gold_dim_depot d
LEFT JOIN gold_fact_rental r
    ON d.DEPOT_CODE = r.depot_code
    AND r.is_returned = 0
GROUP BY d.DEPOT_NAME
ORDER BY currently_out DESC
"""

print("\n========== (f) MACHINES CURRENTLY OUT ==========")
result_f = spark.sql(query_f)
result_f.show(100, truncate=False)


# ============================================================
# FINAL RECONCILIATION CHECKS
# ============================================================

print("\n========== RECONCILIATION ==========")

print("Gold rental rows:", spark.table("gold_fact_rental").count())
print("Gold billing rows:", spark.table("gold_fact_billing").count())

print(
    "Total currently-out rentals:",
    spark.sql("""
        SELECT COUNT(*) AS currently_out
        FROM gold_fact_rental
        WHERE is_returned = 0
    """).first()["currently_out"]
)

print(
    "Total revenue across depots:",
    spark.sql(query_d).agg(
        F.sum("revenue").alias("total_revenue")
    ).first()["total_revenue"]
)


StatementMeta(, bdb537af-1d12-4b47-9705-783769169944, 22, Finished, Available, Finished, False)


Schema: gold_fact_rental
root
 |-- rental_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- depot_code: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- checkout_ts: timestamp (nullable = true)
 |-- checkin_ts: timestamp (nullable = true)
 |-- rental_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- is_returned: integer (nullable = true)
 |-- rental_duration_days: integer (nullable = true)
 |-- asset_days_in_window: integer (nullable = true)


Schema: gold_fact_billing
root
 |-- BILL_ID: string (nullable = true)
 |-- RENTAL_ID: string (nullable = true)
 |-- BILL_DATE: string (nullable = true)
 |-- AMOUNT_INR: decimal(12,2) (nullable = true)
 |-- PAYER_TYPE: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- depot_code: string (nullable = true)
 |-- asset_